
# Ghost in the Aether — Event Simulator

Generates 400 real-time security and communications events across 4 scripted milestones.
Publishes to Eventhouse via Azure Event Hub connection string.

**Configuration:**
- The Event Hub connection string is injected automatically by `deploy.ps1` from the `AetherES` Eventstream (no manual setup). Optionally set `AETHER_EVENTHUB_CONNECTION_STRING` to override for a manual run.
- Adjust `SIMULATION_SPEED` and `NOISE_RATE` for demo pacing.
- Set `SCRIPTED_ONLY = True` to stream only the scripted milestone beats (no noise).



In [ ]:

%pip install azure-eventhub==5.11.6



In [ ]:

import json
import logging
import os
import random
import time
import traceback
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone

import pandas as pd
from azure.eventhub import EventData, EventHubProducerClient, TransportType

logging.getLogger("azure.eventhub").setLevel(logging.WARNING)
print("Imports loaded")



In [ ]:

# Populated automatically by deploy.ps1 (the {{...}} placeholder is replaced with the
# live Eventstream connection string). For manual runs, set the env var to override.
CONNECTION_STRING = os.getenv("AETHER_EVENTHUB_CONNECTION_STRING", "") or "{{EVENTHUB_CONNECTION_STRING}}"
if CONNECTION_STRING.startswith("{{"):
    CONNECTION_STRING = ""

# Simulator controls
SIMULATION_MODE = "COMPRESSED"  # REALTIME | COMPRESSED
SIMULATION_SPEED = 50.0  # 1.0 = real-time pacing
UPDATE_INTERVAL_SECONDS = 27
TOTAL_EVENTS = 400
BATCH_SIZE = 1
NOISE_RATE = 0.35  # 0.0 to 1.0
SCRIPTED_ONLY = False  # True = stream only the scripted milestone beats, no noise
SEED = 42
SCENARIO_ID = "aether-night-001"

# Narrative roster
CHARACTERS = {
    1: "Julian Croft",
    2: "Evelyn Reed",
    3: "Marcus Thorne",
    4: "Anya Sharma",
    5: "Dr. Alistair Finch",
}

LOCATIONS = {
    101: "Julian's Study",
    102: "Server Room",
    103: "Evelyn's Suite",
    104: "Marcus's Suite",
    105: "Back Path",
    106: "Dr. Finch's Suite",
}

if not CONNECTION_STRING:
    raise ValueError(
        "AETHER_EVENTHUB_CONNECTION_STRING is empty. Set it before running the stream."
    )

random.seed(SEED)
RUN_ID = datetime.now(timezone.utc).strftime("run-%Y%m%d-%H%M%S")
SIM_START = datetime.now(timezone.utc)

print(f"Run ID: {RUN_ID}")
print(f"Scenario: {SCENARIO_ID}")
print(f"Total events: {TOTAL_EVENTS} | Batch size: {BATCH_SIZE}")
print(f"Mode: {SIMULATION_MODE} | Speed: {SIMULATION_SPEED}x")
print(f"Scripted only: {SCRIPTED_ONLY}")



In [ ]:

SCRIPTED_MILESTONES = [
    {
        "offset": 0,
        "security": {
            "PersonID": 1,
            "LocationID": 101,
            "EventType": "Access Granted",
        },
        "comms": {
            "CommsType": "Email",
            "Sender": "Evelyn Reed",
            "Recipient": "Julian Croft",
            "Subject": "RE: Code Ownership",
            "Body": "We need to discuss what happened to my original architecture.",
            "Status": "Sent",
        },
    },
    {
        "offset": 45,
        "security": {
            "PersonID": 3,
            "LocationID": 101,
            "EventType": "Movement Detected",
        },
        "comms": {
            "CommsType": "TextMessage",
            "Sender": "Marcus Thorne",
            "Recipient": "Julian Croft",
            "Subject": "",
            "Body": "Stop ignoring me. We settle this tonight.",
            "Status": "Draft",
        },
    },
    {
        "offset": 75,
        "security": {
            "PersonID": 2,
            "LocationID": 102,
            "EventType": "Server Access",
        },
        "comms": {
            "CommsType": "LogEntry",
            "Sender": "AetherSystem",
            "Recipient": "OpsTeam",
            "Subject": "Server Access Alert",
            "Body": "Unexpected late-night server room access observed.",
            "Status": "Archived",
        },
    },
    {
        "offset": 150,
        "security": {
            "PersonID": 5,
            "LocationID": 101,
            "EventType": "Door Unlock",
        },
        "comms": {
            "CommsType": "PhoneCall",
            "Sender": "Security",
            "Recipient": "OpsTeam",
            "Subject": "Emergency",
            "Body": "Body discovered in study. Lockdown initiated.",
            "Status": "Delivered",
        },
    },
]

PERSON_HOME = {
    1: 101,  # Julian
    2: 103,  # Evelyn
    3: 104,  # Marcus
    4: 105,  # Anya
    5: 106,  # Finch
}

ACCESS_CONTROL = {
    1: {101, 102, 105},
    2: {103, 102, 105},
    3: {104, 105, 101},
    4: {105, 103},
    5: {106, 101, 102},
}

ROUTINE_LOCATIONS = {
    1: [101, 101, 102, 105],
    2: [103, 103, 105, 102],
    3: [104, 104, 105, 101],
    4: [105, 105, 103],
    5: [106, 106, 101, 102],
}

SENSITIVE_LOCATIONS = {101, 102}

SECURITY_EVENT_WEIGHTS = [
    ("Access Granted", 0.42),
    ("Door Unlock", 0.16),
    ("Movement Detected", 0.19),
    ("Network Activity", 0.11),
    ("Power State Change", 0.07),
    ("Alarm Triggered", 0.05),
]

COMM_TYPES = ["Email", "TextMessage", "Slack", "PhoneCall", "LogEntry"]


@dataclass
class StreamEvent:
    event_id: str
    event_type: str
    payload: dict


def _event_time_for(index: int) -> datetime:
    return SIM_START + timedelta(seconds=index * UPDATE_INTERVAL_SECONDS)


def _weighted_choice(weighted_values: list[tuple[str, float]]) -> str:
    values = [v for v, _ in weighted_values]
    weights = [w for _, w in weighted_values]
    return random.choices(values, weights=weights, k=1)[0]


PERSON_LAST_LOCATION = PERSON_HOME.copy()


def _make_security_noise_body(person_id: int) -> tuple[dict, bool]:
    previous_location = PERSON_LAST_LOCATION[person_id]
    allowed_locations = ACCESS_CONTROL[person_id]
    routine_locations = ROUTINE_LOCATIONS[person_id]

    # Most events follow routine movement; occasionally probe a sensitive area.
    if random.random() < 0.18:
        location_id = random.choice(list(SENSITIVE_LOCATIONS))
    else:
        location_id = random.choice(routine_locations)

    denied = (location_id not in allowed_locations) or (random.random() < 0.04)

    if denied:
        body = {
            "PersonID": person_id,
            "LocationID": location_id,
            "EventType": "Access Denied",
        }
        suspicious = location_id in SENSITIVE_LOCATIONS
        return body, suspicious

    if location_id == previous_location and random.random() < 0.55:
        event_type = "Movement Detected"
    elif location_id == 102 and random.random() < 0.45:
        event_type = "Server Access"
    else:
        event_type = _weighted_choice(SECURITY_EVENT_WEIGHTS)

    PERSON_LAST_LOCATION[person_id] = location_id
    suspicious = (
        location_id in SENSITIVE_LOCATIONS
        and event_type in {"Door Unlock", "Server Access", "Alarm Triggered"}
        and person_id != 1
    )
    body = {
        "PersonID": person_id,
        "LocationID": location_id,
        "EventType": event_type,
    }
    return body, suspicious


def make_security_event(index: int, scripted: dict | None) -> StreamEvent:
    event_time = _event_time_for(index)

    if scripted:
        body = scripted
        suspicious = True
        if "PersonID" in body and "LocationID" in body:
            PERSON_LAST_LOCATION[body["PersonID"]] = body["LocationID"]
    else:
        # Favor suspects over the victim for ambient movement noise.
        person_id = random.choices([2, 3, 4, 5, 1], weights=[0.26, 0.24, 0.20, 0.18, 0.12], k=1)[0]
        body, suspicious = _make_security_noise_body(person_id)

    payload = {
        "event_type": "SecurityLogs",
        "event_id": f"{RUN_ID}-sec-{index}",
        "run_id": RUN_ID,
        "scenario_id": SCENARIO_ID,
        "event_time_utc": event_time.isoformat(),
        "ingest_time_utc": datetime.now(timezone.utc).isoformat(),
        "Timestamp": event_time.isoformat(),
        "Suspicious": suspicious,
        **body,
    }

    return StreamEvent(payload["event_id"], "SecurityLogs", payload)


def make_comms_event(index: int, scripted: dict | None) -> StreamEvent:
    event_time = _event_time_for(index)

    if scripted:
        body = scripted
    else:
        sender_id = random.choice(list(CHARACTERS.keys()))
        recipient_id = random.choice(list(CHARACTERS.keys()))
        body = {
            "CommsType": random.choice(COMM_TYPES),
            "Sender": CHARACTERS[sender_id],
            "Recipient": CHARACTERS[recipient_id],
            "Subject": "Routine check-in",
            "Body": "Background operational traffic.",
            "Status": random.choice(["Sent", "Delivered", "Archived"]),
        }

    payload = {
        "event_type": "Communications",
        "event_id": f"{RUN_ID}-com-{index}",
        "run_id": RUN_ID,
        "scenario_id": SCENARIO_ID,
        "event_time_utc": event_time.isoformat(),
        "ingest_time_utc": datetime.now(timezone.utc).isoformat(),
        "Timestamp": event_time.isoformat(),
        "Suspicious": scripted is not None,
        **body,
    }

    return StreamEvent(payload["event_id"], "Communications", payload)


def build_event_pairs(total_events: int) -> list[StreamEvent]:
    scripted_by_offset = {m["offset"]: m for m in SCRIPTED_MILESTONES}
    out: list[StreamEvent] = []

    for i in range(total_events):
        milestone = scripted_by_offset.get(i)

        # Scripted-only mode: emit just the milestone beats, skipping all noise.
        if SCRIPTED_ONLY and not milestone:
            continue

        scripted_security = milestone["security"] if milestone else None
        scripted_comms = milestone["comms"] if milestone else None

        out.append(make_security_event(i, scripted_security))

        # Hybrid mode: add communication noise based on NOISE_RATE or milestone.
        if scripted_comms or (not SCRIPTED_ONLY and random.random() < NOISE_RATE):
            out.append(make_comms_event(i, scripted_comms))

    return out


events = build_event_pairs(TOTAL_EVENTS)
print(f"Prepared {len(events)} events for publish")

# Preview a few events for schema sanity.
pd.DataFrame([e.payload for e in events[:3]])



In [ ]:

producer = EventHubProducerClient.from_connection_string(
    CONNECTION_STRING,
    transport_type=TransportType.AmqpOverWebsocket,
    retry_total=3,
)


def publish_events(all_events: list[StreamEvent]) -> tuple[int, int]:
    sent = 0
    failed = 0

    # Keep a tiny pacing delay for near-real-time playback during demos.
    sleep_s = UPDATE_INTERVAL_SECONDS / SIMULATION_SPEED if SIMULATION_MODE else 0

    for i in range(0, len(all_events), BATCH_SIZE):
        batch_events = all_events[i : i + BATCH_SIZE]
        event_batch = producer.create_batch()

        try:
            for e in batch_events:
                event_batch.add(EventData(json.dumps(e.payload)))

            producer.send_batch(event_batch)
            sent += len(batch_events)

        except Exception:
            failed += len(batch_events)
            traceback.print_exc()

        if sent > 0 and sent % 100 == 0:
            print(f"Progress: sent={sent} failed={failed}")

        if sleep_s > 0:
            time.sleep(sleep_s)

    return sent, failed


start_ts = time.time()
try:
    sent_count, failed_count = publish_events(events)
finally:
    producer.close()

elapsed = time.time() - start_ts
print("Stream complete")
print(f"Run ID: {RUN_ID}")
print(f"Sent: {sent_count} | Failed: {failed_count}")
print(f"Elapsed: {elapsed:.2f}s")
if elapsed > 0:
    print(f"Throughput: {sent_count / elapsed:.2f} events/sec")

